# Jour 3 · Comprendre l'interface Streamlit


## Une interface, pas un nouveau moteur RAG

Streamlit transforme un script Python en application web locale. Il affiche des composants et réagit aux interactions, mais la recherche, les tools et l'agent doivent rester dans un cœur Python séparé et testable.

Streamlit se lance dans un terminal ; il ne s'exécute pas à l'intérieur de ce notebook.


In [ ]:
import sys
from pathlib import Path


def find_course_root() -> Path:
    """Retrouve la racine, même si Jupyter démarre dans un sous-dossier."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "documents").is_dir() and (candidate / "jour_01").is_dir():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable.")


ROOT = find_course_root()
DOCUMENTS_DIR = ROOT / "documents"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

# Les imports Python ne suivent pas automatiquement les chemins de fichiers.
# Cette ligne permet notamment d'importer le package local `apps` depuis un
# notebook lancé dans jour_03 ou corrections/jour_03.
root_text = str(ROOT)
if root_text not in sys.path:
    sys.path.insert(0, root_text)

print("Racine du cours :", ROOT)


In [ ]:
import streamlit as st

print("Version Streamlit :", st.__version__)
print("Application finale :", ROOT / "apps" / "app_helpdesk.py")


## Le cycle de rerun

À chaque message ou clic, Streamlit relance le script du haut vers le bas. Une variable Python ordinaire est donc recréée.

Ce fonctionnement est simple, mais il oblige à conserver explicitement l'état utile et les ressources coûteuses.


![Le script Streamlit est réexécuté après une interaction](../assets/jour_01/12_cycle_streamlit.png)

*Un rerun reconstruit l'interface ; session state et cache conservent ce qui doit survivre.*


## `st.session_state` conserve l'état de la session

L'historique affiché et l'identifiant de conversation appartiennent à une session utilisateur. Le pattern d'initialisation est toujours le même : créer la clé seulement si elle n'existe pas.

```python
if "messages" not in st.session_state:
    st.session_state.messages = []
```

Une session n'est pas une base de données durable : fermer la session peut faire disparaître cet état.


### À vous de jouer — isoler la logique de l'historique

Écrivez une fonction pure qui ajoute un message. Elle sera testable sans serveur Streamlit.


In [ ]:
# À vous de jouer — retournez une nouvelle liste contenant le message.
def ajouter_message(historique, role, contenu):
    return list(historique)


test = ajouter_message([], "user", "Bonjour")
print(test)


## `@st.cache_resource` conserve les ressources lourdes

Le modèle, les embeddings, la connexion Chroma ou l'agent coûtent cher à reconstruire. Le cache de ressources permet de les réutiliser entre les reruns.


![Session state, cache et variables ordinaires](../assets/jour_01/12_state_et_cache.png)

*L'historique et les ressources lourdes n'ont ni la même durée de vie ni le même niveau de partage.*


```python
@st.cache_resource
def get_agent():
    return build_agent()
```

Une ressource globale peut être partagée entre plusieurs sessions. Elle doit donc être sûre pour cet usage. Dans une application sensible, l'isolation des utilisateurs et des conversations doit être conçue explicitement.


## Les composants de chat

- `st.chat_message("user")` ou `("assistant")` crée une bulle ;
- `st.chat_input(...)` attend le prochain message ;
- une boucle sur l'historique redessine les anciens messages après chaque rerun.

```python
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

question = st.chat_input("Votre question")
```


## Le streaming est une amélioration d'affichage

Le streaming montre progressivement les morceaux produits par le modèle. Il réduit l'impression d'attente, mais ne diminue pas forcément le temps de calcul.

```python
def morceaux():
    for chunk in llm.stream(messages):
        if chunk.content:
            yield chunk.content

reponse = st.write_stream(morceaux())
```

Nous gardons un `spinner` dans l'application finale pour simplifier l'observation des tools et des sources. Le streaming pourra être ajouté après validation fonctionnelle.


## Séparer interface et cœur métier

Le fichier `helpdesk_core.py` construit la base et l'agent. Le fichier `app_helpdesk.py` gère seulement l'affichage, la session et les interactions.

Cette séparation permet de tester le cœur sans démarrer un navigateur.


In [ ]:
import ast


core_path = ROOT / "apps" / "helpdesk_core.py"
app_path = ROOT / "apps" / "app_helpdesk.py"

for path in [core_path, app_path]:
    source = path.read_text(encoding="utf-8")
    ast.parse(source)
    print(path.name, "→ syntaxe valide ·", len(source.splitlines()), "lignes")

app_source = app_path.read_text(encoding="utf-8")
for attendu in ["st.session_state", "st.cache_resource", "st.chat_input", "st.chat_message"]:
    print(attendu, "→", attendu in app_source)


## Lancer l'application

Fermez éventuellement les serveurs Streamlit précédents, puis double-cliquez sur `LANCER_APP.cmd` à la racine du cours.

Équivalent dans PowerShell :

```powershell
.venv\Scripts\python.exe -m streamlit run appspp_helpdesk.py
```

## À retenir

Streamlit réexécute, `session_state` se souvient d'une session, `cache_resource` réutilise les objets lourds et le cœur métier doit rester indépendant de l'interface.
